# ACE-Step: uji konteks dan penyambungan bass

Gunakan **VM Colab CUDA yang sudah punya checkout bridge/runtime/bobot dari notebook 10**. Jika perlu, salin sel-sel ini ke notebook pada VM yang masih aktif. Tool ZIP ini bukan installer model. Upload `ace_context_tools.zip` dan piano asli saat diminta.

Semua flag inferensi awalnya **False**. Jalankan uji 90 detik dahulu setelah memeriksa hasil 60 detik; 120 detik hanya jika perlu. Ini eksperimen, bukan bukti bahwa chunking sudah memperbaiki musik. Hasil 60 detik lama belum dianalisis di chat ini.


In [ ]:
from pathlib import Path
import importlib.util, json, sys, tempfile, uuid, zipfile
from google.colab import files

for dependency in ("numpy", "scipy", "soundfile"):
    assert importlib.util.find_spec(dependency), f"Kernel notebook memerlukan {dependency}; jangan ubah environment model pinned"

ROOT_HINT = globals().get("ROOT")
print("Upload ace_context_tools.zip dan piano asli dalam satu pilihan")
uploaded = files.upload()
packages = [n for n in uploaded if Path(n).name == "ace_context_tools.zip"]
audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
assert len(packages) == 1 and len(audios) == 1, "Pilih satu tool ZIP dan satu piano asli"
TOOLS = Path(tempfile.mkdtemp(prefix="ace_context_tools_"))
zip_path = TOOLS / "tools.zip"
zip_path.write_bytes(uploaded[packages[0]])
with zipfile.ZipFile(zip_path) as archive:
    for name in ("ace_context_chunks.py", "electric_full_song.py"):
        (TOOLS / name).write_bytes(archive.read(name))
sys.path.insert(0, str(TOOLS))
spec = importlib.util.spec_from_file_location("ace_context_uploaded", TOOLS / "ace_context_chunks.py")
context = importlib.util.module_from_spec(spec)
spec.loader.exec_module(context)
ROOT = context.single.discover_root(Path.cwd(), ROOT_HINT)
if ROOT is None:
    raise RuntimeError("Checkout bridge belum ada di VM ini. Setup notebook 10 dahulu; jangan menganggap upload tool sudah menyediakan model/GPU")
SOURCE = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
SOURCE.write_bytes(uploaded[audios[0]])
OUT = ROOT / "runs" / ("ace_context_" + uuid.uuid4().hex[:10])
plan = context.prepare(SOURCE, OUT, context=60, margin=6, fade=.58)
PLAN_PATH = OUT / "plan.json"
print("ROOT:", ROOT, "| PLAN:", PLAN_PATH)
for chunk in plan["chunks"]:
    print(chunk["index"], "input model:", chunk["model_duration"], "detik", "core:", round((chunk["core_end"]-chunk["core_start"])/32000, 3))
print("Tanpa SongMap terverifikasi: batas ini bukan downbeat atau bar yang diketahui")


## Satu probe durasi

Default `90`, kemudian `120` jika 90 bersih dan memori cukup. `60` juga tersedia untuk kontrol baru. Caption, seed, FP32, CFG dan langkah tetap sama. Dengarkan bagian waktu yang sama, jangan hanya melihat status generated. Jangan menjalankan beberapa probe sekaligus di T4 yang sama.


In [ ]:
from IPython.display import Audio, display

DURATION = 90  # 60 / 90 / 120
EXECUTE_PROBE = False  # True hanya pada runtime CUDA/bobot yang siap
PROBE_OUT = OUT / (f"probe_{DURATION}_" + uuid.uuid4().hex[:8])
probe_report = context.single.generate(ROOT, OUT / "source.wav", PROBE_OUT,
    start=0, duration=DURATION, execute=EXECUTE_PROBE)
print("Status:", probe_report["status"], "|", PROBE_OUT)
if EXECUTE_PROBE:
    display(Audio(filename=str(PROBE_OUT / "mix_preview.mp3"), normalize=False))


In [ ]:
# Jalankan setelah probe selesai, atau setelah gagal untuk report yang tersedia.
feedback = PROBE_OUT / "full_song_feedback.zip"
if feedback.is_file():
    files.download(str(feedback))
else:
    print("Belum ada feedback inferensi. Probe dry-run tidak membuat hasil model.")


## Chunk seluruh lagu, setelah durasi konteks dipilih

Plan awal membatasi **seluruh konteks model menjadi 60 detik termasuk margin**. Jika memilih cap lain, buat plan baru dengan `context.prepare(SOURCE, NEW_OUT, context=90, margin=6, fade=.58)` dan perbarui `PLAN_PATH`. Jangan mengganti plan yang telah memiliki hasil.

Generasi sequential dan memuat model lagi per chunk. Margin/reload menambah waktu. Chunk selesai dipakai kembali setelah verifikasi; proses diffusion yang gagal diulang sebagai attempt baru. Seed sama tidak menjamin timbre/register/groove sama.


In [ ]:
EXECUTE_CHUNKS = False
if EXECUTE_CHUNKS:
    progress = context.run_chunks(ROOT, PLAN_PATH)
    report = context.stitch_run(PLAN_PATH)
    print("Sambungan perlu didengar:", [s["index"] for s in report["seams"] if s["manual_review"]])
    display(Audio(filename=str(PLAN_PATH.parent / "stitched/mix_full.mp3"), normalize=False))
else:
    print("Tidak menjalankan inferensi atau stitching; plan tersimpan di", PLAN_PATH)


In [ ]:
# Paket kecil: mix/bass MP3, report seam dan cuplikan seam; tanpa bobot model.
stitched = PLAN_PATH.parent / "stitched"
if (stitched / "seams.json").is_file():
    bundle = PLAN_PATH.parent / "chunked_feedback.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(PLAN_PATH, "plan.json")
        for path in sorted(stitched.iterdir()):
            if path.suffix in {".json", ".mp3"} or path.name.startswith("seam_"):
                archive.write(path, "stitched/" + path.name)
    files.download(str(bundle))
else:
    print("Belum ada hasil stitching lengkap.")


Equal-power crossfade **tetap dapat membatalkan fase bass**. Helper mencari kandidat berdasarkan energi/korelasi, membuang tepi model, dan melaporkan perubahan level/pitch; ia tidak menjamin sambungan musikal. Birama/downbeat tidak ditebak dari BPM. Repaint dan conditioning BPM/key belum diterapkan. Lihat `ace-context-chunks.md` di tool ZIP untuk rincian.
